In [ ]:
pip install langchain_community langchain_text_splitters langchain_google_genai faiss-cpu langchain_core langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

os.environ["GOOGLE_API_KEY"] = "AIzaSyD7Z6KBAril9wgsgRPBqmm3Ro2Wn7aB2bdhfdvhjbhdhvdvgdbgvhdsbgfb7k"

In [ ]:

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
no_rag_response = llm.invoke("leave policy")
print("WITHOUT RAG:")
print(no_rag_response.content)
print("\n" + "="*60 + "\n")



WITHOUT RAG:
A **leave policy** is a formal document or set of guidelines established by an organization to define the rules, procedures, and entitlements related to employee absences from work. Its primary purpose is to ensure fairness, consistency, and compliance with labor laws while managing employee time off effectively.

A comprehensive leave policy typically covers various types of leave, including:

1.  **Paid Time Off (PTO) / Annual Leave / Vacation Leave:**
    *   **Purpose:** For rest, recreation, and personal matters.
    *   **Accrual:** How employees earn leave days (e.g., X days per month/year, based on tenure).
    *   **Usage:** How to request, notice period required, maximum consecutive days.
    *   **Carryover:** Whether unused leave can be carried over to the next year, and if so, how much.
    *   **Payout:** Whether unused leave is paid out upon termination.

2.  **Sick Leave:**
    *   **Purpose:** For illness, injury, or medical appointments for the employee o

In [ ]:
loader = TextLoader("company_policy.txt")
documents = loader.load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
improved_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

improved_chunks = improved_splitter.split_documents(documents)

In [ ]:
embeddings_model = GoogleGenerativeAIEmbeddings(
    model="text-embedding-004"
)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

/tmp/ipykernel_2431/793128546.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# pip install faiss-cpu

In [ ]:
vector_db = FAISS.from_documents(
    documents=improved_chunks,
    embedding=embeddings_model
)

In [ ]:
retriever = vector_db.as_retriever(search_kwargs={"k": 7})

In [ ]:
template = """You are an HR assistant. Use the following pieces of context
to answer the question. If the answer is not in the context, say you don't know.

Context: {context}

Question: {question}

Answer:"""

QA_PROMPT = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

improved_qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | QA_PROMPT
    | llm
    | StrOutputParser()
)

In [ ]:
query = "Explain leave policy"
response = improved_qa_chain.invoke(query)
print("WITH IMPROVED RAG:")
print(response)

WITH IMPROVED RAG:
At TechNova Solutions, the leave policy includes:

*   **Annual Leave:** Full-time employees receive 20 days of paid annual leave per calendar year, accrued monthly. Up to 5 days of unused annual leave can be carried over to the next calendar year.
*   **Sick Leave:** Employees are provided with 10 days of paid sick leave annually for personal illness or family medical care.
*   **Parental Leave:** New parents (for birth or adoption) are offered 12 weeks of fully paid parental leave.
*   **Bereavement Leave:** 3 days of paid leave are given in the event of the death of an immediate family member.
*   **Approval Process:** All leave requests must be submitted via the HR portal at least two weeks in advance.
